# 03 – Pandas Analysis
## Reel Insights: Global Movie Industry (TMDB)

This notebook answers 10 analytical questions using Pandas and the six cleaned CSV datasets.

## 1. Import libraries and load cleaned datasets

In [ ]:
import os
import pandas as pd
import numpy as np

PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CLEANED_DIR = os.path.join(DATA_DIR, "cleaned")

movies = pd.read_csv(os.path.join(CLEANED_DIR, "movies_cleaned.csv"))
genres = pd.read_csv(os.path.join(CLEANED_DIR, "genres_cleaned.csv"))
movie_genres = pd.read_csv(os.path.join(CLEANED_DIR, "movie_genres_cleaned.csv"))
cast = pd.read_csv(os.path.join(CLEANED_DIR, "cast_cleaned.csv"))
crew = pd.read_csv(os.path.join(CLEANED_DIR, "crew_cleaned.csv"))
movie_keywords = pd.read_csv(os.path.join(CLEANED_DIR, "movie_keywords_cleaned.csv"))

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["release_year"] = movies["release_date"].dt.year
print("Loaded cleaned datasets.")

## Q1 – Unique movies, genres, cast members and crew members

In [ ]:
unique_counts = pd.DataFrame({
    "entity": ["Movies", "Genres", "Cast Members", "Crew Members"],
    "unique_count": [movies.movie_id.nunique(), genres.genre_id.nunique(),
                     cast.person_id.nunique(), crew.person_id.nunique()]
})
display(unique_counts)

## Q2 – Missing or zero budget/revenue

In [ ]:
zb, zr = movies.budget.eq(0), movies.revenue.eq(0)
quality = pd.DataFrame({
    "measure": ["Zero budget", "Zero revenue", "Both zero",
                "Missing budget", "Missing revenue"],
    "count": [zb.sum(), zr.sum(), (zb & zr).sum(),
              movies.budget.isna().sum(), movies.revenue.isna().sum()]
})
display(quality)
display(movies[zb | zr | movies.budget.isna() | movies.revenue.isna()]
        [["movie_id","title","budget","revenue"]])

## Q3 – Profit and ROI for movies with valid financial data

In [ ]:
valid_financial = movies[(movies.budget > 0) & (movies.revenue > 0)].copy()
valid_financial["profit"] = valid_financial.revenue - valid_financial.budget
valid_financial["roi_percentage"] = valid_financial.profit / valid_financial.budget * 100
profit_roi = valid_financial[["movie_id","title","budget","revenue","profit","roi_percentage"]]
print("Valid financial records:", len(profit_roi))
display(profit_roi.sort_values("profit", ascending=False))

## Q4 – Top 15 ROI, budget at least $1 million

In [ ]:
top_15_roi = (valid_financial[valid_financial.budget >= 1_000_000]
              .nlargest(15, "roi_percentage")
              [["title","budget","revenue","profit","roi_percentage"]])
display(top_15_roi.reset_index(drop=True))

## Q5 – Movies released per year and peak year

In [ ]:
movies_per_year = (movies.dropna(subset=["release_year"])
                   .groupby("release_year").movie_id.nunique()
                   .sort_values(ascending=False))
display(movies_per_year.rename("movie_count").to_frame())
print("Peak year:", movies_per_year.idxmax(), "| Movies:", movies_per_year.max())

## Q6 – Runtime outliers using IQR

In [ ]:
runtime = movies.runtime.dropna()
q1, q3 = runtime.quantile(.25), runtime.quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
runtime_outliers = movies[(movies.runtime < lower) | (movies.runtime > upper)]
print("Q1:", q1, "Q3:", q3, "IQR:", iqr, "Bounds:", lower, upper)
print("Outlier count:", len(runtime_outliers))
display(runtime_outliers[["movie_id","title","runtime"]].sort_values("runtime"))

## Q7 – Primary genre and average popularity

In [ ]:
# The dataset has no explicit genre-order field. Lowest genre_id is an approximation.
primary = (movie_genres.sort_values(["movie_id","genre_id"])
           .drop_duplicates("movie_id", keep="first"))
primary_data = (primary.merge(genres, on="genre_id")
                .merge(movies[["movie_id","popularity"]], on="movie_id"))
primary_popularity = (primary_data.groupby("genre_name")
    .agg(average_popularity=("popularity","mean"),
         movie_count=("movie_id","nunique"))
    .sort_values("average_popularity", ascending=False))
display(primary_popularity)

## Q8 – Top 10 prolific actors and average movie rating

In [ ]:
actor_ratings = cast.merge(movies[["movie_id","vote_average"]], on="movie_id")
top_actors = (actor_ratings.groupby(["person_id","actor_name"])
    .agg(movie_count=("movie_id","nunique"),
         average_movie_rating=("vote_average","mean"))
    .sort_values(["movie_count","actor_name"], ascending=[False,True])
    .head(10).reset_index())
display(top_actors)

## Q9 – Repeated movie titles and release years

In [ ]:
repeated_titles = (movies.groupby("title")
    .agg(record_count=("movie_id","nunique"),
         release_years=("release_year",
             lambda s: ", ".join(str(int(x)) for x in sorted(s.dropna().unique()))))
    .query("record_count > 1").sort_values("record_count", ascending=False)
    .reset_index())
display(repeated_titles)
print("Repeated titles:", len(repeated_titles))

## Q10 – Unusual popularity relative to vote count

In [ ]:
# Fit a simple linear trend in log space and inspect standardized residuals.
p = movies[(movies.popularity > 0) & (movies.vote_count > 0)].copy()
p["log_votes"] = np.log1p(p.vote_count)
p["log_popularity"] = np.log1p(p.popularity)
slope, intercept = np.polyfit(p.log_votes, p.log_popularity, 1)
p["residual"] = p.log_popularity - (intercept + slope*p.log_votes)
p["residual_zscore"] = p.residual / p.residual.std(ddof=1)
unusual = p[p.residual_zscore.abs() >= 2].copy()
unusual["pattern"] = np.where(unusual.residual_zscore > 0,
    "Popularity higher than expected", "Popularity lower than expected")
display(unusual[["title","vote_count","popularity","residual_zscore","pattern"]]
        .sort_values("residual_zscore", ascending=False))
print("Unusual observations:", len(unusual))

## 12. Overall Pandas summary

This notebook analyses dataset size, financial data quality, profit and ROI, release trends, runtime outliers, genre popularity, actor appearances, repeated titles and unusual popularity patterns.